# Week 1 – Extraction harness walkthrough
Supporting notebook: schema mapping, rules reference run and error taxonomy. Production logic lives in `src/irm/`. Run `python run.py week1` first.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
from irm import config

In [2]:
from irm.schema import FIELD_MAP
pd.DataFrame(FIELD_MAP, columns=['eval_field','source','column','comparison','notes'])

,eval_field,source,column,comparison,notes
0,employee_code,profile,Code,exact,Primary matching key
1,full_name,profile,Fullname,normalised,Fallback matching key
2,position,manifest,Position,normalised,Title label + primary stack
3,current_project,manifest,CurrentProject,normalised,First item of Project
4,primary_stack,profile,Techstack[0],normalised,First item of Techstack
5,total_experience_years,profile,CompanyTenure // 12,exact,Months -> whole years
6,business_domain,profile,Domain,normalised,
7,education.degree_level,profile,Education,normalised,Derived from degree wording
8,education.major,profile,Major,normalised,
9,education.school,profile,School,normalised,


In [3]:
s1 = json.loads((config.EXTRACTION_DIR/'w1_textlayer_v1_all_digital'/'_summary.json').read_text())
s2 = json.loads((config.EXTRACTION_DIR/'w1_textlayer_v2_all_digital'/'_summary.json').read_text())
pd.DataFrame({'v1': {g: m['f1'] for g, m in s1['by_group'].items()}, 'v2': {g: m['f1'] for g, m in s2['by_group'].items()}})

,v1,v2
business_domain,1.0000,1.0
certificates,1.0000,1.0
current_project,1.0000,1.0
education,0.8979,1.0
employee_code,1.0000,1.0
full_name,1.0000,1.0
languages,1.0000,1.0
position,1.0000,1.0
primary_stack,1.0000,1.0
projects,1.0000,1.0


In [4]:
slots = pd.read_csv(config.EXTRACTION_DIR/'w1_textlayer_v1_all_digital'/'_slots.csv')
slots[slots.error.notna()].head(9)

,code,field,expected,predicted,outcome,error
184,20030,education.degree_level,associate degree,NaN,FN,missing
186,20030,education.major,artificial intelligence,associate degree,FPFN,misplaced
187,20030,education.school,university of technology education ho chi minh...,artificial intelligence in university of techn...,FPFN,wrong_value
248,20034,education.degree_level,phd,NaN,FN,missing
250,20034,education.major,electrical engineering,phd,FPFN,misplaced
251,20034,education.school,ho chi minh city university of information tec...,electrical engineering in ho chi minh city uni...,FPFN,wrong_value
442,20064,education.degree_level,phd,NaN,FN,missing
444,20064,education.major,information technology,phd,FPFN,misplaced
445,20064,education.school,ton duc thang university,information technology in ton duc thang univer...,FPFN,wrong_value


## Gemini runs
After `python run.py compare-tiers --prompt v1`, load `docs/evaluation/tier_comparison_v1.csv` here.